In [ ]:
import torch, time
import numpy as np
from PIL import Image, ImageDraw
import cv2
import os
from pathlib import Path
import json
from transformers import AutoModel, AutoTokenizer, AutoProcessor
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

C:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def inference_examp1e(model_path, image_path, output_dir=None):
    """
    학습된 모델로 추론 예제
    
    Args:
        model_path (str): 학습된 모델 경로
        image_path (str): 추론할 이미지 경로 (파일 또는 폴더)
        output_dir (str): 결과 저장 경로 (None이면 기본 경로 사용)
    """
    
    print(f"\n=== 추론 예제 ===")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # 출력 디렉토리 설정
    if output_dir is None:
        output_dir = "runs/predict"  # YOLO 기본 경로
    
    # 출력 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"모델: {model_path}")
    print(f"입력: {image_path}")
    print(f"출력 경로: {output_dir}")

    try:
        model = YOLO(model_path)
        
        # 추론 실행 - 저장 경로 지정
        results = model(
            image_path, 
            device=device, 
            conf=0.6,            # 신뢰도 경계값
            imgsz=640,
            save=True,           # 결과 이미지 저장
            save_txt=True,       # 텍스트 라벨 저장
            save_conf=True,      # 신뢰도 포함
            save_crop=True,      # 감지된 객체 크롭 저장 (선택적)
            project=output_dir,  # 저장 경로 지정
            name="inference",    # 실행별 폴더명
            exist_ok=True       # 기존 폴더 덮어쓰기 허용
        )
        
        # 결과 출력
        print(f"\n=== 추론 결과 ===")
        for i, r in enumerate(results):
            # 기본 정보
            boxes_count = len(r.boxes) if r.boxes is not None else 0
            masks_count = len(r.masks) if r.masks is not None else 0
            
            print(f"이미지 {i+1}:")
            print(f"  - 감지된 객체: {boxes_count}개")
            print(f"  - 분할된 객체: {masks_count}개")
            
            # 클래스별 상세 정보
            if r.boxes is not None and len(r.boxes) > 0:
                class_names = [
                    'solid_yellow_lane', 'dotted_yellow_lane', 'double_yellow_lane',
                    'crosswalk', 'sidewalk', 'firehydrant', 'car', 'license_plate'
                ]
                
                detected_classes = {}
                for box in r.boxes:
                    class_id = int(box.cls)
                    conf = float(box.conf)
                    class_name = class_names[class_id] if class_id < len(class_names) else f"class_{class_id}"
                    
                    if class_name not in detected_classes:
                        detected_classes[class_name] = []
                    detected_classes[class_name].append(conf)
                
                print("  - 클래스별 감지 결과:")
                for class_name, confidences in detected_classes.items():
                    avg_conf = sum(confidences) / len(confidences)
                    print(f"    {class_name}: {len(confidences)}개 (평균 신뢰도: {avg_conf:.3f})")
        
        # 저장된 파일 경로 출력
        save_dir = Path(output_dir) / "inference"
        if save_dir.exists():
            print(f"\n=== 저장된 파일 ===")
            print(f"결과 저장 경로: {save_dir}")
            
            # 저장된 파일 목록 출력
            saved_files = list(save_dir.glob("*"))
            if saved_files:
                for file_path in sorted(saved_files):
                    file_size = file_path.stat().st_size / 1024  # KB 단위
                    print(f"  - {file_path.name} ({file_size:.1f} KB)")
            else:
                print("  저장된 파일이 없습니다.")
        
        return results, str(save_dir)
        
    except Exception as e:
        print(f"추론 중 오류 발생: {str(e)}")
        return None, None

In [2]:
def measure_inference_fps(model_path, image_paths):
    """
    FPS 측정용 함수
    
    Args:
        model_path (str): 학습된 모델 경로 (.pt)
        image_paths (list[str]): 추론할 이미지 경로 리스트

    Returns:
        fps (float): 초당 프레임 수
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = YOLO(model_path)

    print(f"FPS 측정 시작: 총 {len(image_paths)}장 이미지")
    start = time.time()
    
    for image_path in image_paths:
        _ = model(image_path, device=device, verbose=False)  # 추론 결과 저장 없이 수행
    
    end = time.time()
    total_time = end - start
    fps = len(image_paths) / total_time if total_time > 0 else 0

    print(f"총 시간: {total_time:.3f}초")
    print(f"FPS (Frames Per Second): {fps:.2f}")
    return fps


In [26]:
def main():
    # 1. 경로 입력
    source_images_dir = 'C:/Users/User/Downloads/dataset/images'
    source_labels_dir = 'C:/Users/User/Downloads/dataset/labels'
    changed_labels_dir = 'C:/Users/User/Downloads/dataset/change_labels'

    model_path = hf_hub_download(repo_id="won3956/parking_gaurd", filename="best.pt")

    # 10. 추론 예제 (선택사항) - 수정된 부분
    print("\n=== 추론 테스트 ===")   
    
    # 단일 이미지 테스트
    test_image = 'C:/Users/User/Downloads/dataset/images/IMG_2214.JPG'
    if test_image and os.path.exists(test_image):
        output_folder = 'C:/Users/User/Desktop/Parking_Gaurd/output_dir'        
        inference_example(model_path, test_image, output_folder)
    else:
        print("이미지 경로가 잘못되었습니다.")

In [ ]:
def main2():
    image_folder = 'C:/Users/User/Downloads/midnight/images'
    model_path = hf_hub_download(repo_id="won3956/parking_gaurd", filename="best.pt")   # 11X
    
    image_paths = [
        os.path.join(image_folder, f)
        for f in os.listdir(image_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    measure_inference_fps(model_path,image_paths)


In [10]:
main2()

FPS 측정 시작: 총 251장 이미지
총 시간: 23.068초
FPS (Frames Per Second): 10.88
